Enhancing Reasoning Capability in LLMs
===


**Created by Dr Chao Shu (chao.shu@qmul.ac.uk)**

Submitted By

**Name**:

**QMUL ID**:

**BUPT ID**:

# Introduction
---

In this project, you are expected to comprehensively apply the reasoning and inference techniques you have learned in L01, L2 and L05 (L03 & L06 were covered in the Coding Challenge) in this module to unlock the reasoning potential of a small-scale non-reasoning model. You will build an end-to-end pipeline to fine-tune the model to improve its reasoning capability and benchmark its reasoning performance on the GSM8k dataset. This notebook will guide to you to:

1. **Load** a pre-trained instruct model and attach a LoRA adapter for efficient fine-tuning.
2. **Prepare** the GSM8K training dataset into the chat-message format required by `GRPOTrainer`.
3. **Design reward functions** that guide the model toward producing well-formatted, correct Chain-of-Thought reasoning.
4. **Train** the model using GRPO with your reward functions. 
5. **Evaluate** the fine-tuned model on the full GSM8K test set.

## Prerequisites

Set Up the Virtual Environment on [EECS JupyterHub (JHub)](https://hub.comp-teach.qmul.ac.uk/)

1. Log in to [EECS JHub](https://hub.comp-teach.qmul.ac.uk/) using your QMUL credentials. Search and start the "EBU6505 Reasoning and Agents" instance.
2. Follow Step 3 to Step 7 in the README.md in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education) to set up and activate the virtual environment on EECS JHub. You can clone either the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education) or your forked repo for this project. If you clone your forked repo, please make sure you haven't changed the `pyproject.toml` file and `uv.lock` file, i.e., you haven't installed/upgraded any packages in your forked repo. 
3. Install an interpreter for Jupyter notebooks. Make sure your venv is activated before running the command.

    ```bash
    python -m ipykernel install --user --name reasoning-llms
    ```

4. Upload this notebook to the root of the repo.
5. Before you run the notebook, make sure you choose `reasoning-llms` as the kernel.

> ‼️ Important Note:
> 
>  **The notebook you submit will be tested under the virtual environment specified in the `pyproject.toml` file and `uv.lock` file in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education). Please make sure you strictly follow the instructions to set up a virtual environment specified in the `pyproject.toml` file and `uv.lock` file in the [reasoning-and-agents-education repo](https://github.com/chaoshu1201/reasoning-and-agents-education). If your notebook fails during tests due to inconsistent virtual environment, you will receive 0 mark for the project.**
> 
> Although the same `pyproject.toml` file and `uv.lock` file are used to set up virtual environments on your laptop and JHub, the environments are different. On JHub, where we use Linux and NVIDIA GPUs, `unsloth` and `vllm` will be installed for memory-efficient fine-tuning and high-throughput inference, respectively.

## Project Instructions

In this project:
- **Restricted Environment**: You will work in a restricted environment. Your notebook/script can only use the pre-downloaded `Qwen2-1.5B-Instruct` model and the `GSM8k` dataset.
- **Limited Hardware Resources**: You will work with limited hardware resources. You have a maximum of **50 GB storage*** space and **20 GB GPU VRAM****. This constraint mirrors the real-world limitations of industrial LLM development. You need to know the gap between theory and practice, how the libraries/tools you use are designed in a way that allows you complete your tasks with limited hardware resources.
- **Time Constraints**: Fine-tuning and evaluating LLMs are computationally intensive tasks, even on high-performance GPUs. **Do not delay your project execution until the deadline is imminent**, as training runs and comprehensive evaluations require significant time to complete.
- **Strict Compliance**: You **MUST** follow the instructions in this notebook:
    - You must use certain settings/configurations if instructed in the notebook.
    - You must use certain functions if instructed in the notebook.

  While GenAI can easily generate code from scratch, professional environments typically require you to contribute to a complex, existing codebase. You are expected to implement features within the provided framework rather than rewriting the system, a critical skill for working in big tech companies.
- Language Standard: All responses and code comments within the notebook MUST be written in English. Use of other languages will result in a loss of marks.
- You are expected to refer to the [GRPOTrainer](https://huggingface.co/docs/trl/v1.3.0/en/grpo_trainer#grpo-trainer) documents to learn and understand the usage of this library (with the help from AI) during the development of the project, as what you will do in your own project and in your future career. Although you are also encouraged to use GenAI assistants and/or coding agents to help with the development, you should use critical thinking and verify the solutions generated by AI against relevant documents, to ensure you fully understand the solutions provided by AI.

> 💡 *Note:
>
> The initial storage space is 5 GB and will be expanded automatically based on your usage. When your remaining storage is lower than 30% of your current space, your storage will be added by another 5 GB in a couple of minutes.
> 
> Before you start you project, we recoommend you mannually expand your storage to at least 40 GB. You can upload a large files (e.g., 4.5 GB) to your space, wait for 5 minutes, duplicate that file (e.g., use `cp` command: `cp your_large_file your_large_file_1`). Keep duplicate the large file for 7 times, you should get a storage space of 40 GB. After that you can delete all the files you upload and duplicate. The storage size will ramain unchanged.

> 💡 **Note:
>
> Every student should be allocated to a GPU slice with 20 GB VRAM. The GPU types may vary. Sometimes, you will get more than 20 GB VRAM due to a race condition on the server side (Lucky you!). If that happens, it is recommended that you stop your server and/or log out, relaunch the server to get a 20 GB VRAM GPU slice. You can choose to enjoy your luck with more VRAM in the current server instance, but you need to bear in mind that your recipe must work with less than 20 GB VRAM and your recipe will be tested with 20 GB VRAM. Exceeding the 20 GB VRAM limitation will invalidate your recipe and evaluation results during marking.

**Keep your JHub server alive when training and evaluation for a long time**

When you plan to perform a training and evaluation that will take a long time (e.g., > 10 hrs) and you need to leave the JHub page and cannot keep the JHub page open for such a long time, you may need to keep the server alive when you leave the JHub page by using [jupyter-keepalive](https://github.com/minrk/jupyter-keepalive).

1. Deactivate your uv venv. (make sure your uv venv is NOT activated)
2. Run the following command to install `jupyter-keepalive` **in the system environment**.

    ```bash
    pip install jupyter-keepalive
    ```

3. Stop and log out the server, then log back in.
4. Follow the [How to Use](https://github.com/minrk/jupyter-keepalive#how-to-use) instructions of `jupyter-keepalive`.
5. Once a keep-alive session is active, a countdown timer appears in the bottom status bar of JupyterLab showing the remaining time. When no keep-alive session is running, the timer is hidden.

Good Practice
- Only use the keep-alive feature when necessary (e.g. long-running jobs or temporary breaks).
- Remember to stop the timer when you’re finished to free up system resources.

## Project Objectives

The objective of this project is to comprehensively apply the reasoning techniques you have learned through L01 to L05 (except L03 & L04) to improve the reasoning capability of the `Qwen2-1.5B-Instruct` model, so that:
1. The improved model can correctly solve as many problems in the GSM8k dataset as possible (**improve the mean problem-solving rate $\bar{R_s}$**).
2. The improved model can output reasoning and answers following the exact format below (**improve the mean correct reasoning format rate $\bar{R_f}$**).

   *Expected Output Format:*
   ```text
   <think>[Your step-by-step reasoning process here]</think>
   <answer>[Final numerical answer here]</answer>
   ```

   *Example:*
   ```text
   "<think> Janet's ducks lay 16 eggs per day. She eats three for breakfast, so she has 16 - 3 = 13 eggs left. She bakes muffins with 4 eggs, so she has 13 - 4 = 9 eggs left. She sells the remaining eggs at $2 per egg, so she makes 9 * $2 = $18 every day at the farmers' market. </think>\n<answer>18</answer>"
   ```

## Marking Rubric

Your **project mark $S$ (UK Scale)** will be calculated as:

$$ S = B + \bar{R_f} \times 10 + min\big(max((\bar{R_s} - 0.60), 0) \times 100 \times 4, 60\big) $$

Where:

$B \in [0, 30]$ is the mark for your notebook (code) report. Most of them are not performance-relevant. Marks will be awarded as long as the code and answers are correct and reasonable.

$\bar{R_f} \in [0, 1]$ is the mean correct format rate in your evaluation results.

$\bar{R_s} \in [0, 1]$ is the mean problem-solving rate in your evaluation results.

How $\bar{R_s}$ and $\bar{R_f}$ are calculated: Because generative LLMs exhibit variance in their outputs (especially when sampling with temperature), your model is evaluated over $N$ independent rounds (In this project, we use $N=5$) across the entire GSM8k test set to ensure a robust and fair assessment.

**Mean Problem-Solving Rate ($\bar{R_s}$)**: For each evaluation round $i$, a solve rate $R_{s,i}$ is calculated as the proportion of test problems where the model's final extracted numerical answer correctly matches the ground truth. The overall mean is the average across all $N$ rounds: $$ \bar{R_s} = \frac{1}{N} \sum_{i=1}^{N} R_{s,i} $$

**Mean Correct Format Rate ($\bar{R_f}$)**: For each evaluation round $i$, a format rate $R_{f,i}$ is calculated as the proportion of model responses that successfully comply with the required structural tags (i.e., containing both the <think>...</think> reasoning block and the <answer>...</answer> solution block). The overall mean is the average across all $N$ rounds: $$ \bar{R_f} = \frac{1}{N} \sum_{i=1}^{N} R_{f,i} $$

# PART I: GRPO Fine-Tuning
---

## Imports and Seed Setup

You may add more imports if necessary (but not to install new libraries). You MUST add all imports in the following cell.

In [ ]:
import os
import re
import json
import csv
import time
import random
from datetime import datetime

import numpy as np
from tqdm import tqdm

import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from vllm import SamplingParams
from trl import GRPOTrainer, GRPOConfig
from transformers import TrainerCallback

## Student ID and Recipe Configuration

Set your Student ID and configure the experiment recipe dictionary `recipe_dic`. The `recipe_dic` packs all experiment hyper-parameters into a single dictionary so that every setting is captured in one place.

> 💡 Note:
> 
> - You **MUST** set your `STUDENT_ID` below.
> - Do **NOT** change the `seed` value – a fixed seed ensures fair comparison across students.

⛔ DO NOT change the next cell

In [ ]:
# Set random seeds for reproducibility
seed = 3407

# Model path
model_name = os.path.expandvars("$HOME/reasoning-and-agents-shared/huggingface/models/Qwen2-1.5B-Instruct")

# Output directory for fine-tuned lora adapter and evaluation results
output_dir = "./grpo_output"
os.makedirs(output_dir, exist_ok=True)

# Set global random seeds for reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

Please set your QMUL student ID

In [ ]:
# TODO: !!IMPORTANT!! Set your QMUL Student ID
STUDENT_ID = "221154321"  # Replace with your QMUL student ID

Please set your experiment parameters. For the provided values, you can change them when you are testing your project, but you MUST change them back to the original values in your final run for submission. 

> 💡 Note:
>
> You don't need to build a complete recipe at the beginning. You can add more hyperparameters and tweak their values during the optimisation of your model.

In [ ]:
# TODO: Experiment recipe – all hyper-parameters in one dictionary
recipe_dic = {
    "identity_config": {
        "student_id": STUDENT_ID, 
    },
    "model_config": {
        "model":          model_name,
        "lora_dir":       None,         # filled in inside run_eval_for_targets
        # Add more model-related configurations
    },
    "dataset_config": {
        # train_size and eval_size filled in after dataset loading
    },
    "grpo_config": {
        "seed":                        seed,
        "report_to":                   "none",
        "output_dir":                  os.path.join(output_dir, "checkpoints"),
        # Add more GRPO-related configurations
    },
    "eval_config": {
        "num_rounds":  5,
        "seed":        seed,
        # Set eval_model to "base", "lora", or "both" depending on which model(s) you want to evaluate
        # For example, set eval_model = "base" to evaluate only the original base model, 
        # or eval_model = "lora" to evaluate only the fine-tuned LoRA model. 
        # Setting eval_model = "both" will evaluate both and save results separately.
        "eval_model":  "both",
        "eval_target": None,         # filled in per iteration inside run_eval_for_targets
    },
    "sampling_params": {
        # Add more model-related configurations
    },
}

## Load the Model

In this section, you will load a pre-trained instruct model Qwen2-1.5B-Instruct using Unsloth's `FastLanguageModel` and attach a LoRA (Low-Rank Adaptation) adapter for parameter-efficient fine-tuning.

In this project, we use [Unsloth](https://unsloth.ai/) because it can make fine-tuning LLMs faster and more memory-efficient, which is especially useful when VRAM is limited.

**[Q1.1]** Load the model and attach the LoRA adapter using the parameters from `recipe_dic`. **[4 Marks]**

In [ ]:
print("[1/4] Loading model...")

# TODO: Load the pre-trained model and set parameters as needed.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    fast_inference=True,
    # Set other parameters as needed
)

# TODO: Attach LoRA adapter to the model. Please specify other parameters as needed.
model = FastLanguageModel.get_peft_model(
    model,
    use_gradient_checkpointing="unsloth",
    random_state=seed,
    # Set other parameters as needed
)

print("  Model loaded.\n")

Data Preparation
---

In this section, you will:

1. Define **reasoning and solution tags** (`<think>`, `</think>`, `<answer>`, `</answer>`) that structure the model's output format.
2. Create a **system prompt** instructing the model to reason step-by-step and wrap its output in the defined tags.
3. Load the **[GSM8K dataset](https://huggingface.co/datasets/openai/gsm8k)** and preprocess the training split into the chat-message format expected by `GRPOTrainer`:
   - Each example should have a `"prompt"` field (list of system + user messages) and an `"answer"` field (the ground-truth numerical answer extracted from `"#### X"`).

Firstly, let's define **reasoning and solution tags** (`<think>`, `</think>`, `<answer>`, `</answer>`) that structure the model's output format.

⛔ DO NOT change the next cell

In [ ]:
# Define reasoning_start, reasoning_end, solution_start, solution_end tags
reasoning_start = "<think>"
reasoning_end = "</think>"
solution_start = "<answer>"
solution_end = "</answer>"

**[Q1.2]** Create the system prompt by using the defined the reasoning/solution tags. **[2 Marks]**

In [ ]:
# TODO: Create system_prompt using the tags
system_prompt = \
f"""
Your prompt here
"""

Before we load the GSM8k dataset, let's define a helper function to extract the answer after "####". You must use the `extract_hash_answer()` helper function when necessary.

⛔ DO NOT change the next cell

In [ ]:
def extract_hash_answer(text):
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

Next, we load the GSM8K dataset, preprocess the training split into chat-message format. It is implemented for you.

In [ ]:
print("[2/4] Preparing dataset...")

gsm8k = load_dataset("openai/gsm8k", "main")

train_dataset = gsm8k["train"].map(lambda x: {
    "prompt": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": x["question"]},
    ],
    "answer": extract_hash_answer(x["answer"]),
})

recipe_dic["dataset_config"]["train_size"] = len(train_dataset)

print("  Dataset prepared.\n")

## Reward Functions

In this section, you will design **reward functions** that the `GRPOTrainer` uses to score each generated completion. These rewards shape the model's learning signal.

You should define **regex patterns** to match the expected output format and extract answers from the outputs, then implement reward functions that encourage both correct formatting and correct answers, aiming to achieve the objectives defined in the "**Project Objectives**" section. A large proportion of your project marks is determined by how well the objectives are achieved, as defined in the "**Marking Rubric**" section. 

Each reward function receives `completions` (a list of generated responses) and optionally `prompts` and `answer` (ground truth), and returns a list of numerical scores.

> 💡 Note:
> 
> You are free to design your own reward functions and scoring schemes. The number of reward functions, their names, and the reward values are all up to you. This is a key part of your recipe design.

A Reference definition of regex patterns for format matching and answer extraction are provided in the next cell for you to use in this project. You are free to define your own regex pattern, but your fine-tuned model and recipe will be evaluated using the following regrex patterns.

⛔ DO NOT change the next cell

In [ ]:
# Define regex pattern(s) to match the full <think>...</think><answer>...</answer> format
match_format = re.compile(
    rf"^[\s]{{0,}}"
    rf"{re.escape(reasoning_start)}.+?{re.escape(reasoning_end)}.*?"
    rf"{re.escape(solution_start)}(.+?){re.escape(solution_end)}"
    rf"[\s]{{0,}}$",
    flags=re.MULTILINE | re.DOTALL
)

# Define regex pattern(s) to extract numerical answers from <answer> tags
match_numbers = re.compile(
    re.escape(solution_start) + r".*?([\d\.\,]{1,})",
    flags=re.MULTILINE | re.DOTALL
)

**[Q1.3]** Implement your reward functions. You should implement **at least two** reward functions: one for format checking and one for answer correctness. You may add more. Each reward function must follow the signature expected by [`GRPOTrainer`](https://huggingface.co/docs/trl/v1.3.0/en/grpo_trainer#using-a-custom-reward-function). **[8 Marks]**

In [ ]:
print("[3/4] Setting up reward functions...")

# TODO: Implement your reward functions
# At minimum:
#   - A format-checking reward function
#   - An answer-correctness reward function
# You may add additional reward functions as part of your recipe design

print("  Reward functions defined.\n")

## GRPO Training

In this section, you will:

1. **Configure `GRPOConfig`** with the training hyper-parameters from `recipe_dic`.
2. **Create the `GRPOTrainer`** with your model, tokenizer, reward functions, training args, dataset, and the CSV callback.
3. **Run training** and time the process.
4. **Save the LoRA adapter** for later evaluation and submission.

Before configuring the trainer, let's implement the `CSVLoggingCallback` class. The callback will log general training metrics. (To make it easier and consistent in marking across all students, we don't use Weights & Biases in this project.)

⛔ DO NOT change the next cell

In [ ]:
# CSV logging callback
training_log_path = os.path.join(output_dir, "training_stats.csv")

class CSVLoggingCallback(TrainerCallback):
    def __init__(self, path):
        self.path = path
        self._file = None
        self._writer = None

    def on_train_begin(self, args, state, control, **kwargs):
        self._file = open(self.path, "w", newline="")
        self._writer = csv.DictWriter(self._file, fieldnames=[
            "step", "epoch", "loss", "learning_rate", "grad_norm",
            "reward", "reward_std",
            "completion_length", "kl",
        ])
        self._writer.writeheader()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and self._writer:
            row = {
                "step": state.global_step,
                "epoch": round(state.epoch, 4) if state.epoch else "",
                "loss": logs.get("loss", ""),
                "learning_rate": logs.get("learning_rate", ""),
                "grad_norm": logs.get("grad_norm", ""),
                "reward": logs.get("reward", ""),
                "reward_std": logs.get("reward_std", ""),
                "completion_length": logs.get("completion_length", ""),
                "kl": logs.get("kl", ""),
            }
            self._writer.writerow(row)
            self._file.flush()

    def on_train_end(self, args, state, control, **kwargs):
        if self._file:
            self._file.close()


**[Q1.4]** Configure `GRPOConfig`, create `GRPOTrainer`, run training, and save the LoRA adapter. **[5 Marks]**

> 💡 Note:
>
> - Use `recipe_dic` values for all hyper-parameters.
> - Pass your list of reward functions to `GRPOTrainer`.
> - Save the LoRA adapter to `{output_dir}/grpo_saved_lora/` using `model.save_lora()`.

In [ ]:
print("[4/4] Starting GRPO training...")

# Build grpo_config_kwargs dict from recipe_dic.
grpo_kwargs = {k: v for k, v in recipe_dic["grpo_config"].items() if v is not None}

# TODO: Create GRPOConfig
training_args = "Your code here"

# TODO: Create GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        # TODO: Add your reward functions here
    ],
    args=training_args,
    train_dataset=train_dataset,
    callbacks=[CSVLoggingCallback(training_log_path)],
)


# TODO: Run the GRPO training and time it
t0 = time.time()
# Your code here
elapsed = time.time() - t0
print(f"  GRPO training done. Time: {elapsed:.1f}s")
print(f"  Training stats saved to: {training_log_path}")

# TODO: Save LoRA adapter
lora_dir = os.path.join(output_dir, "grpo_saved_lora")
# Your code here
print(f"  LoRA adapter saved to: {lora_dir}\n")

# Part II: Evaluation
---

In this section, you will evaluate the GRPO-fine-tuned model on the **full GSM8K test set** over multiple rounds. This involves:

1. **Complete the implementation**: Most of the evaluation functions are provided, except for a few parts of `solve_single_problem()` — a function that generates completions for a single problem using unsloth's `fast_generate`. You are required to complete the implementation of the `solve_single_problem()` function.
2. **Running the evaluation** with the relevant parameters from `recipe_dic`. The provided evaluation functions are the same functions that will be used in marking. By running evaluations using these functions, you can have a good estimation of the marks you can get based on your evaluation results. 
3. **Saving results** — Results and logs required for submission will be automatically saved after you run the evaluations.

Let's first define the function to extract the answer from the model output. You MUST use this function in `solve_single_problem()` to extract the answer from the outputs of the model.

⛔ DO NOT change the next cell

In [ ]:
# Define the function to extract the answer from the model output
def extract_answer(output):
    fmt_match = match_format.search(output)
    if fmt_match:
        return fmt_match.group(1).strip(), True   # (answer, has_format)
    num_match = match_numbers.search(output)
    if num_match:
        return num_match.group(1).strip(), False
    return None, False

**[Q2.1]** Complete the `solve_single_problem()` function following the TODO instructions. **[5 Marks]**

In [ ]:
def solve_single_problem(problem, prompt_template, llm, sampling_params, lora_request=None):
    """Solve a single problem in the dataset.
    Arguments:
        problem (dict): The problem to solve.
        prompt_template (str): The prompt to be used as the system message.
        llm: The language model to use.
        sampling_params (dict): Parameters for sampling.
        lora_request (LoRARequest, optional): The LoRA request for fine-tuned evaluation.
            lora_request=None: Inference using the base model; pass a loaded LoRA request for fine-tuned evaluation.
    """

    question = problem["question"]
    expected = extract_hash_answer(problem["answer"])

    if expected is None:
        return None

    messages = [
        {"role": "system", "content": prompt_template},
        {"role": "user", "content": question},
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    try:
        # TODO: Create SamplingParams object from your recipe's sampling parameters
        sp = "Your code here"
        completions = llm.fast_generate(text, sampling_params=sp, lora_request=lora_request, use_tqdm=False)[0].outputs
        text_responses = "Your code here"

        # TODO: Obtain the predicted answer and whether the output has the correct format using extract_answer function
        predicted = "Your code here"
        has_format = "Your code here"

        # --- Correctness check ---
        is_correct = False
        if predicted is not None and expected is not None:
            try:
                is_correct = float(predicted.replace(",", "")) == float(expected.replace(",", ""))
            except (ValueError, TypeError):
                is_correct = predicted.strip() == expected.strip()

        # TODO: Calculate thinking-token count
        think_re = re.compile(
            re.escape(reasoning_start) + r"(.*?)" + re.escape(reasoning_end), re.DOTALL
        )
        
        n_thinking_tokens = "Your code here"

        return {
            "question": question,
            "expected": expected,
            "predicted": predicted,
            "correct": is_correct,
            "has_format": has_format,
            "n_thinking_tokens": n_thinking_tokens,
            "response": text_responses,
        }
    except Exception as e:
        print(f"Error processing example: {e}")
        return None

Define other evaluation functions.

⛔ DO NOT change the next cell

In [ ]:
def run_evaluation(llm, problems, prompt_template, rounds, sampling_params, lora_request=None):
    """Run multi-round evaluation.

    lora_request=None → base model; pass a loaded LoRA request for fine-tuned evaluation.
    """
    all_metrics = []
    all_responses = []

    print(f"\nRunning {rounds} rounds of evaluation...")

    for round_num in range(1, rounds + 1):
        print(f"\n--- Evaluation Round {round_num}/{rounds} ---")
        t0 = time.time()

        results = []
        correct = 0
        format_ok = 0
        total_thinking_tokens = 0

        for i, example in enumerate(tqdm(problems, desc=f"Round {round_num}")):
            res = solve_single_problem(example, prompt_template, llm, sampling_params, lora_request=lora_request)
            if res is None:
                continue
            
            res["index"] = i
            results.append(res)
            
            if res["correct"]:
                correct += 1
            if res["has_format"]:
                format_ok += 1
            total_thinking_tokens += res["n_thinking_tokens"]

        elapsed = time.time() - t0
        n = len(results)
        round_summary = {
            "round": round_num,
            "eval_size": n,
            "correct": correct,
            "solve_rate": round(correct / n, 4) if n else 0,
            "format_rate": round(format_ok / n, 4) if n else 0,
            "avg_thinking_tokens": round(total_thinking_tokens / n, 1) if n else 0,
            "elapsed_seconds": round(elapsed, 1)
        }
        all_metrics.append(round_summary)
        all_responses.append(results)
        
        print(f"  Round {round_num}: solve_rate={round_summary['solve_rate']:.4f} "
              f"({round_summary['correct']}/{round_summary['eval_size']}), "
              f"format_rate={round_summary['format_rate']:.4f}, "
              f"avg_thinking_tokens={round_summary['avg_thinking_tokens']}, "
              f"time={elapsed:.1f}s")

    solve_rates = [r["solve_rate"] for r in all_metrics]
    format_rates = [r["format_rate"] for r in all_metrics]
    thinking_tokens = [r["avg_thinking_tokens"] for r in all_metrics]

    aggregate_summary = {
        "solve_rate": {
            "mean": round(float(np.mean(solve_rates)), 4),
            "std_dev": round(float(np.std(solve_rates)), 4),
            "min": round(float(np.min(solve_rates)), 4),
            "max": round(float(np.max(solve_rates)), 4),
            "per_round": solve_rates,
        },
        "format_rate": {
            "mean": round(float(np.mean(format_rates)), 4),
            "std_dev": round(float(np.std(format_rates)), 4),
            "per_round": format_rates,
        },
        "avg_thinking_tokens": {
            "mean": round(float(np.mean(thinking_tokens)), 1),
            "std_dev": round(float(np.std(thinking_tokens)), 1),
            "per_round": thinking_tokens,
        },
    }
    
    return aggregate_summary, all_metrics, all_responses

def run_eval_for_targets(
    eval_model,
    output_dir,
    model,
    lora_dir,
    test_dataset,
    system_prompt,
    recipe_dic,
):
    """Run evaluation for one or more model variants (base / lora / both).

    Args:
        eval_model:          "base", "lora", or "both".
        output_dir:          Root directory for all outputs.
        model:               Loaded FastLanguageModel (vLLM engine).
        lora_dir:            Path to the saved LoRA adapter directory.
        test_dataset:        HuggingFace dataset split to evaluate on (e.g. gsm8k["test"]).
        system_prompt:       system prompt string.
        recipe_dic:          Full config dict (identity_config, model_config,
                             dataset_config, grpo_config, eval_config,
                             sampling_params). Used directly as the "config"
                             block in the saved metrics JSON.
    """
    print(f"  Test set size: {len(test_dataset)}")
    recipe_dic["dataset_config"]["eval_size"] = len(test_dataset)
    recipe_dic["model_config"]["lora_dir"] = lora_dir

    num_eval_rounds = recipe_dic["eval_config"]["num_rounds"]
    sampling_params = recipe_dic["sampling_params"]
    student_id      = recipe_dic["identity_config"]["student_id"]
    model_name      = recipe_dic["model_config"]["model"]

    # ── Iterate over eval targets ─────────────────────────────────────────────
    eval_targets = ["base", "lora"] if eval_model == "both" else [eval_model]

    for eval_target in eval_targets:
        responses_dir = os.path.join(output_dir, f"responses_{eval_target}")
        os.makedirs(responses_dir, exist_ok=True)

        if eval_target == "base":
            lora_request = None
            print("  Evaluating base model (lora_request=None).")
        else:
            if not os.path.isdir(lora_dir):
                raise FileNotFoundError(
                    f"LoRA directory not found: {lora_dir}. "
                    "Run training first or point output_dir to an existing LoRA run."
                )
            lora_request = model.load_lora(lora_dir)
            print(f"  Evaluating LoRA model from: {lora_dir}")

        aggregate_summary, all_rounds, all_responses = run_evaluation(
            llm=model,
            problems=test_dataset,
            prompt_template=system_prompt,
            rounds=num_eval_rounds,
            sampling_params=sampling_params,
            lora_request=lora_request,
        )

        for round_num, (round_summary, round_results) in enumerate(zip(all_rounds, all_responses), 1):
            round_filename = f"round_{round_num}_responses.json"
            with open(os.path.join(responses_dir, round_filename), "w") as f:
                json.dump({"summary": round_summary, "results": round_results}, f, indent=2, default=str)

        # ── Consolidated metrics file ─────────────────────────────────────────
        recipe_dic["eval_config"]["eval_target"] = eval_target
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        model_identifier = model_name.replace("/", "-").replace(":", "_")
        metrics_filename = f"metrics_{model_identifier}_{eval_target}_{student_id}_{timestamp}.json"
        metrics_path = os.path.join(output_dir, metrics_filename)

        consolidated = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "config": recipe_dic,
            "results": {
                "summary": aggregate_summary,
                "rounds":  all_rounds,
            }
        }

        with open(metrics_path, "w") as f:
            json.dump(consolidated, f, indent=2, default=str)

        # ── Print final summary ───────────────────────────────────────────────
        print(f"\n{'='*60}")
        print(f"MULTI-ROUND EVALUATION SUMMARY ({num_eval_rounds} rounds)")
        print(f"{'='*60}")
        print(f"  Student ID:          {student_id}")
        print(f"  Model:               {model_name}")
        print(f"  Eval model:          {eval_target}")
        print(f"  Test set size:       {len(test_dataset)}")
        print(f"  Rounds:              {num_eval_rounds}")
        print(f"{'─'*60}")
        print(f"  Solve Rate:          {aggregate_summary['solve_rate']['mean']:.4f} "
              f"± {aggregate_summary['solve_rate']['std_dev']:.4f}")
        print(f"    Per round:         {aggregate_summary['solve_rate']['per_round']}")
        print(f"  Format Rate:         {aggregate_summary['format_rate']['mean']:.4f} "
              f"± {aggregate_summary['format_rate']['std_dev']:.4f}")
        print(f"  Avg Thinking Tokens: {aggregate_summary['avg_thinking_tokens']['mean']:.1f} "
              f"± {aggregate_summary['avg_thinking_tokens']['std_dev']:.1f}")
        print(f"{'='*60}")
        print(f"  Metrics saved to:    {metrics_path}")
        print(f"  Per-round responses: {responses_dir}/")

        # Save summary alongside the LoRA (only when a LoRA was evaluated)
        if lora_request is not None:
            summary_filename = f"eval_summary_{student_id}_{timestamp}.json"
            summary_path = os.path.join(lora_dir, summary_filename)
            with open(summary_path, "w") as f:
                json.dump({
                    "student_id": student_id,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "results": aggregate_summary,
                }, f, indent=2)
            print(f"  Summary also saved to: {summary_path}")

**[Q2.2]** Run the multi-round evaluation. **[2 Marks]**

In [ ]:
lora_dir = os.path.join(output_dir, "grpo_saved_lora")

# TODO: Call run_evaluation() and capture results
# Your code here

# PART III: Analysis and Discussions
---

**[Q3.1]** What reasoning/training techniques have you applied in this project to enhance the reasoning capability of the Qwen2-1.5B-Instruct model on the GSM8k dataset? Based on the metrics result file you obtatined from the evaluations, how much does each technique contribute to the reasoning capability improvement? If you want to determine the contribution to performance improvement from each part, how you plan to conduct an ablation test? (Please learn what is an ablation test from GenAI/Search Engine. You don't need to implement the plan in this project). **[3 Marks]**



*Your answer here*

**[Q3.2]** Based on the training log `training_stats.csv` you obtatined from the evaluations, visualise how the loss and reward changes over training steps. What can you find from the chart? (Use GenAI tools to help you generate the visualisation codes.)  **[1 Mark]**



*Your answer here*

---
# Submission Instructions
---

Please follow the steps below to prepare and submit your project.

## Files to Include

Your submission must include the following files and folders:

| Item | Path |
|------|------|
| Base model evaluation responses | `grpo_output/responses_base/` (entire folder) |
| LoRA model evaluation responses | `grpo_output/responses_lora/` (entire folder) |
| Metrics JSON file(s) | `grpo_output/metrics_*.json` (all `.json` files in the root of `grpo_output/`) |
| Training log | `grpo_output/training_stats.csv` |
| Project notebook | `P01_Enhacing_Reasoning_in_LLMs.ipynb` |

> ⚠️ **Important:** Before archiving, make sure your notebook has been fully executed from top to bottom with all cell outputs visible. Do **not** clear outputs before submission.


## How to Create the Archive

Name your archive file using your **QMUL Student ID** as the suffix:

```
submission_<YOUR_STUDENT_ID>.tar.gz
```

For example, if your student ID is `221154321`, your archive should be named:

```
submission_221154321.tar.gz
```

To create the archive, run the following command from the **root of the repository**:

```bash
tar -czvf submission_<YOUR_STUDENT_ID>.tar.gz \
    grpo_output/responses_base/ \
    grpo_output/responses_lora/ \
    grpo_output/metrics_*.json \
    grpo_output/training_stats.csv \
    P01_Enhacing_Reasoning_in_LLMs.ipynb
```

## Upload

1. Download `submission_<YOUR_STUDENT_ID>.tar.gz` from the JHub server to your laptop.
2. Upload `submission_<YOUR_STUDENT_ID>.tar.gz` to the **Project Submission** link on Moodle before the deadline.

> ‼️ **Double-check your submission after uploading to make sure your upload the correct file.**
